# Objective
Correct needle-to-freezer (NtoF) effects in the preprocessed KP55K NMR data.

In [ ]:
suppressPackageStartupMessages({
  library(here)
  library(tidyverse)
  library(maplet)
  library(ggpubr)
  library(ggrepel)
})


# Load preprocessed data

`NeedleToFreezer_Hrs` is not stored on the preprocessed maplet. It is aligned from `preprocessing_inputs/kp55k_ntof.csv` by sample ID.


In [ ]:
# preprocessed maplet from nmr_preprocessing.ipynb
D <- readRDS(here('preprocessing_inputs/kp55k_maplet_preprocessed.rds'))

# align needle-to-freezer hours to the preprocessed sample order
ntof <- readr::read_csv(
  here('preprocessing_inputs/kp55k_ntof.csv'),
  col_select = c(StudyID, NeedleToFreezer_Hrs),
  show_col_types = FALSE
)

ntof <- ntof[match(colnames(D), ntof$StudyID), ]
stopifnot(identical(ntof$StudyID, colnames(D)))
stopifnot(!anyNA(ntof$NeedleToFreezer_Hrs))

colData(D)$NeedleToFreezer_Hrs <- ntof$NeedleToFreezer_Hrs

dim(D)

# Log2 transform and NtoF bins
Correction is fit on the log2 scale. Bins are used for the phenylalanine example below. Samples with NtoF of 0–34 hours are the reference group for rescaling.

In [ ]:
# define NtoF bins for phenylalanine example
D <- D %>%
  mt_pre_trans_log(base = 2) %>% 
  mt_anno_mutate(
    anno_type = 'samples',
    col_name = 'NtoF_bin',
    # batches match distribution of operational NtoF bins
    term = case_when(
      NeedleToFreezer_Hrs <= 34 ~ '0-34 hrs',
      NeedleToFreezer_Hrs > 34 & NeedleToFreezer_Hrs <= 58 ~ '34-58 hrs',
      NeedleToFreezer_Hrs > 58 & NeedleToFreezer_Hrs <= 82 ~ '58-82 hrs',
      NeedleToFreezer_Hrs > 82 & NeedleToFreezer_Hrs <= 106 ~ '82-106 hrs',
      NeedleToFreezer_Hrs > 106 ~ '106+ hrs',
      TRUE ~ 'AllElse',
      is.na(NeedleToFreezer_Hrs) ~ 'Missing'
    )
  )

# counts per bin 
D %>%
  colData() %>%
  as.data.frame() %>%
  dplyr::count(NtoF_bin) %>%
  arrange(desc(n))

# Rescaled residualization

For each metabolite, fit `log2(abundance) ~ NeedleToFreezer_Hrs` and keep the residuals. Add back the median log2 abundance among samples with needle-to-freezer time in 0–34 hours so corrected values stay on the log2 scale, anchored to the shortest storage-time group.


In [ ]:
# backup and extract NMR data
D1 <- D

assay_data <- assay(D1)

# define ntof variable
batch_effect <- colData(D1)$NeedleToFreezer_Hrs
selected_samples <- which(batch_effect %in% 0:34) # define reference group

# initialize corrected data matrix
corrected_data <- matrix(nrow = nrow(assay_data), ncol = ncol(assay_data))
rownames(corrected_data) <- rownames(assay_data)
colnames(corrected_data) <- colnames(assay_data)

for (i in seq_len(nrow(assay_data))) {
  fit <- lm(assay_data[i, ] ~ batch_effect)
  median_value <- median(assay_data[i, selected_samples], na.rm = TRUE)
  corrected_data[i, ] <- residuals(fit) + median_value
}

assay(D1) <- corrected_data # store corrected data 

# Univariate associations

Pearson correlation between each metabolite and needle-to-freezer hours, before and after correction. FDR is Benjamini–Hochberg.


In [ ]:
# correlation of every metabolite with needle-to-freezer hours
ntof_associations <- function(mat, ntof_hrs) {
  results <- lapply(rownames(mat), function(met) {
    cor_res <- cor.test(mat[met, ], ntof_hrs)
    data.frame(
      metabolite = met,
      pearson_cor = unname(cor_res$estimate),
      p_value = cor_res$p.value
    )
  })
  results <- do.call(rbind, results)
  results$fdr <- p.adjust(results$p_value, method = 'fdr')
  results
}

ntof_hrs <- colData(D)$NeedleToFreezer_Hrs
results_uncorrected <- ntof_associations(assay(D), ntof_hrs)
results_corrected <- ntof_associations(assay(D1), ntof_hrs)

In [ ]:
# volcano of metabolite–NtoF correlations
plot_ntof_volcano <- function(results, subtitle) {
  top_hits <- results %>% arrange(p_value) %>% head(10)
  ggplot(results, aes(x = pearson_cor, y = -log10(fdr))) +
    geom_point(aes(color = fdr < 0.05), alpha = 0.5) +
    geom_hline(yintercept = -log10(0.05), linetype = 'dashed') +
    geom_text_repel(data = top_hits, aes(label = metabolite), box.padding = 0.5, max.overlaps = 10) +
    scale_x_continuous(labels = scales::scientific_format()) +
    labs(
      title = 'Univariate Met-NtoF Associations',
      subtitle = subtitle,
      x = 'Pearson Correlation',
      y = '-log10(FDR)'
    ) +
    theme_pubr() +
    theme(
      plot.title = element_text(hjust = 0.5),
      plot.subtitle = element_text(hjust = 0.5)
    )
}

options(repr.plot.height = 5, repr.plot.width = 10)

# save and export 
p_volcano <- cowplot::plot_grid(
    plot_ntof_volcano(results_uncorrected, 'Uncorrected'),
    plot_ntof_volcano(results_corrected, 'Corrected'),
    ncol = 2
  )

ggsave(here('outputs/figures/NTOF_volcano.png'), p_volcano, width = 10, height = 5)

## Phenylalanine example
Phenylalanine is amongst the most significantly impacted metabolites that increases with longer needle-to-freezer time. The same bins are shown before and after rescaled residualization to manually inspect the ntof correction. 

In [ ]:
# violin of log2(Phe) by NtoF bin, with bin sample size in the axis label
plot_phe_by_bin <- function(se, title) {
  bin_order <- c('0-34 hrs', '34-58 hrs', '58-82 hrs', '82-106 hrs', '106+ hrs')

  tmp_df <- data.frame(
    Phe = assay(se)['Phe', ],
    NtoF_bin = colData(se)$NtoF_bin
  )

  sample_sizes <- tmp_df %>%
    group_by(NtoF_bin) %>%
    summarise(n = n(), .groups = 'drop') %>%
    mutate(
      NtoF_bin = factor(NtoF_bin, levels = bin_order),
      label = paste0(NtoF_bin, '\n(n=', n, ')')
    ) %>%
    arrange(NtoF_bin)

  tmp_df <- tmp_df %>%
    left_join(sample_sizes %>% select(NtoF_bin, label), by = 'NtoF_bin')
  tmp_df$label <- factor(tmp_df$label, levels = sample_sizes$label)

  ggviolin(tmp_df, x = 'label', y = 'Phe', fill = 'label') +
    geom_boxplot(width = 0.3, fill = 'white', color = 'black') +
    labs(title = title, x = '', y = 'log2(Phe)') +
    rotate_x_text(45) +
    theme(plot.title = element_text(hjust = 0.5), legend.position = 'none')
}

options(repr.plot.height = 4, repr.plot.width = 8)

# store and export 
p_phe_drift <- cowplot::plot_grid(
  plot_phe_by_bin(D, 'Uncorrected Phe by NtoF Bin'),
  plot_phe_by_bin(D1, 'Corrected Phe by NtoF Bin'),
  ncol = 2
)

# export
ggsave(here('outputs/figures/NTOF_phe.png'), p_phe_drift, width = 8, height = 4)

In [ ]:
# export individual maplet objects
rowdata <- D1 %>% rowData() %>% as.data.frame()
coldata <- colData(D1) %>% as.data.frame()
assay <- assay(D1) %>% t() %>% as.data.frame() %>% tibble::rownames_to_column("StudyID")

# save for downstream use 
write.csv(rowdata, here('data/rowdata.csv'))
write.csv(coldata, here('data/meta.csv'))
write.csv(assay, here('data/assay.csv'))